# tsdiag CARE v6 benchmark on Kaggle
Attach the `tsdiag-care-kaggle` bundle and the official `CARE_To_Compare.zip` as separate Kaggle Datasets. Start with the smoke run, then use `full` for the publishable benchmark.

In [ ]:
from pathlib import Path
import sys, zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
sources = sorted(INPUT.rglob('src/tsdiag/__init__.py'))
if not sources:
    bundles = sorted(INPUT.rglob('tsdiag-care-kaggle.zip'))
    if len(bundles) != 1:
        raise RuntimeError(f'Expected one tsdiag bundle, found: {bundles}')
    target = WORK / 'tsdiag_bundle'
    with zipfile.ZipFile(bundles[0]) as archive:
        archive.extractall(target)
    sources = sorted(target.rglob('src/tsdiag/__init__.py'))
if len(sources) != 1:
    raise RuntimeError(f'Expected one tsdiag source tree, found: {sources}')
PROJECT_ROOT = sources[0].parents[2]
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
RUNNER = PROJECT_ROOT / 'kaggle' / 'run_care_kaggle.py'
print('Project:', PROJECT_ROOT)
print('Runner:', RUNNER)

In [ ]:
archives = sorted(INPUT.rglob('CARE_To_Compare.zip'))
layouts = sorted({p.parent.parent for p in INPUT.rglob('event_info.csv') if p.parent.name.startswith('Wind Farm ')})
print('Visible CARE archives:', archives)
print('Visible extracted CARE roots:', layouts)
if not archives and not layouts:
    raise RuntimeError('CARE was not found. Attach the CARE v6 Kaggle Dataset before continuing.')

In [ ]:
import subprocess

RUN_MODE = 'smoke'  # change to 'full' after the smoke run passes
VERIFY_MD5 = False  # use True only when CARE_To_Compare.zip itself is visible
command = [sys.executable, str(RUNNER), '--mode', RUN_MODE]
if VERIFY_MD5 and archives:
    command.extend(['--care-path', str(archives[0])])
    command.append('--verify-md5')
elif VERIFY_MD5:
    raise RuntimeError('Kaggle expanded the CARE ZIP, so byte-for-byte ZIP MD5 verification is unavailable.')
subprocess.run(command, check=True)

In [ ]:
import json

output = Path('/kaggle/working/care_results')
manifest = json.loads((output / 'kaggle_run_manifest.json').read_text())
display(manifest['checks'])
display(manifest['summary'])
print('Artifacts:', [p.name for p in output.iterdir()])